In [71]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import os, pickle
import re

import numpy as np
import matplotlib, matplotlib.pyplot as plt
from matplotlib import ticker
import hist
import vector
import pandas as pd

from physics.simulation import msq, mcfm
from physics.hstar import c6
from datasets import coefficient
from models import taylr

import torch
from torch.utils.data import DataLoader
import lightning as L

In [72]:
torch.set_float32_matmul_precision('high')

matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": "", 
})

In [73]:
RUN_DIR = 'run/h2l2v'
TAYLR_DIR = 'taylr'

VERSIONS = ['0', '0', '0', '0']
ncoeffs = len(VERSIONS)

EVENTS_TRAIN_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}','events_train.pkl') for i  in range(ncoeffs)]
EVENTS_VAL_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}','events_val.pkl') for i  in range(ncoeffs)]
EVENTS_TEST_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'events_test.pkl') for i  in range(ncoeffs)]

SCALER_X_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'scaler_X.pkl') for i in range(ncoeffs)]
SCALER_y_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'scaler_y.pkl') for i in range(ncoeffs)]

METRICS_PATHS = [os.path.join(RUN_DIR, TAYLR_DIR+f'_{i+1}', 'lightning_logs', f'version_{VERSIONS[i]}', 'metrics.csv') for i in range(ncoeffs)]

CHECKPOINT_PATHS = []
pattern = re.compile(r'epoch=(\d+)-val_loss=.*\.ckpt')
for i in range(ncoeffs):
    checkpoint_dir = os.path.join(RUN_DIR, TAYLR_DIR + f'_{i+1}', 'lightning_logs', f'version_{VERSIONS[i]}', 'checkpoints')
    all_ckpts = [f for f in os.listdir(checkpoint_dir) if pattern.match(f)]
    
    if not all_ckpts:
        raise FileNotFoundError(f"No matching checkpoint files found in {checkpoint_dir}")
    
    # Sort by extracted epoch number (as int), descending
    all_ckpts.sort(key=lambda f: int(pattern.match(f).group(1)), reverse=True)
    
    latest_ckpt = os.path.join(checkpoint_dir, all_ckpts[0])
    CHECKPOINT_PATHS.append(latest_ckpt)

In [ ]:
BATCH_SIZE = 2048
FEATURES = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
FEATURES = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

In [75]:
with open(EVENTS_TRAIN_PATHS[0], 'rb') as f:
    events_train = pickle.load(f)
with open(EVENTS_VAL_PATHS[0], 'rb') as f:
    events_val = pickle.load(f)
with open(EVENTS_TEST_PATHS[0], 'rb') as f:
    events_test = pickle.load(f)

In [76]:
with open(SCALER_X_PATHS[i], 'rb') as f:
    scaler_X = pickle.load(f)
validation_datasets = [coefficient.CoefficientDataset(events_val, features=FEATURES, coefficient_index=i+1, component=msq.Component.SBI, scaler_X=scaler_X) for i in range(ncoeffs)]
testing_datasets = [coefficient.CoefficientDataset(events_test, features=FEATURES, coefficient_index=i+1, component=msq.Component.SBI, scaler_X=scaler_X) for i in range(ncoeffs)]

In [77]:
metrics = [np.genfromtxt(path, delimiter=',') for path in METRICS_PATHS]

epochs = [m[1::2,0] for m in metrics]
steps = [m[1::2,1] for m in metrics]
train_loss = [m[0::2,2][1:] for m in metrics]
val_loss = [m[1::2,3] for m in metrics]

In [78]:
fig, axes = plt.subplots(2,2, figsize=(8,5), layout='constrained')

for i in range(ncoeffs):
    axes[i//2,i%2].plot(epochs[i][1:], train_loss[i][1:], color='red', label='Training')
    axes[i//2,i%2].plot(epochs[i][1:], val_loss[i][1:], color='royalblue', label='Validation')

    axes[i//2,i%2].legend(frameon=False, fontsize=12)
    axes[i//2,i%2].set_xlabel('Epoch', fontsize=12)
    axes[i//2,i%2].set_ylabel('Loss', fontsize=12)
    axes[i//2,i%2].tick_params(axis='both', labelsize=12)

plt.savefig('taylr_loss.pdf', bbox_inches='tight')
plt.close()

In [79]:
models = [taylr.TAYLR.load_from_checkpoint(path) for path in CHECKPOINT_PATHS]

In [80]:
validation_dataloaders = [DataLoader(val, BATCH_SIZE) for val in validation_datasets]
testing_dataloaders = [DataLoader(val, BATCH_SIZE) for val in testing_datasets]

In [81]:
trainer = L.Trainer(accelerator='gpu', devices=1)

# pred_val = [torch.concatenate(trainer.predict(model, testing_dataloaders[i])).numpy()[:,np.newaxis].flatten() for i, model in enumerate(models)]
pred_test = [torch.concatenate(trainer.predict(model, testing_dataloaders[i])).numpy()[:,np.newaxis].flatten() for i, model in enumerate(models)]

# target_val = [ds.y for ds in validation_datasets]
target_test = [ds.y for ds in testing_datasets]

# weight_val = [ds.w for ds in validation_datasets]
weight_test = [ds.w for ds in testing_datasets]

for i in range(ncoeffs):
    with open(SCALER_y_PATHS[i], 'rb') as f:
        scaler_y = pickle.load(f)

    # pred_val[i] = scaler_y.inverse_transform(pred_val[i].reshape(-1,1)).reshape(-1)
    pred_test[i] = scaler_y.inverse_transform(pred_test[i].reshape(-1,1)).reshape(-1)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


Predicting: |                                                                                                 …

In [82]:
def calibration_curve(axes, p, t, labels, bounds, bins, ylim):
    step_size = (bounds[1]-bounds[0])/bins

    targets_binned = [t[(p > bounds[0]+step_size*i) & (p <= bounds[0]+step_size*(i+1))] for i in range(bins)]

    targets_mean = np.array([np.mean(bin) for bin in targets_binned])
    targets_dev = np.array([np.std(bin) for bin in targets_binned])

    centers = np.array([bounds[0]+(i+1/2)*step_size for i in range(bins)])

    formatter_1 = ticker.ScalarFormatter(useMathText=True)
    formatter_1.set_scientific(True)
    formatter_1.set_powerlimits((-2,2))

    formatter_2 = ticker.ScalarFormatter(useMathText=True)
    formatter_2.set_scientific(True)
    formatter_2.set_powerlimits((-2,2))

    axes[0].set_xticklabels([])

    axes[0].yaxis.set_major_formatter(formatter_1)
    axes[1].yaxis.set_major_formatter(formatter_2)
    axes[1].xaxis.set_major_formatter(formatter_2)

    axes[0].errorbar(centers, centers, color='black', linestyle='--', label='targets')
    axes[0].errorbar(centers, targets_mean, yerr=targets_dev, color='royalblue', marker='o', markersize=4, linestyle='none', label='predictions', alpha=0.8)

    axes[0].set_xlim(*bounds)
    axes[0].set_ylim(*bounds)
    axes[0].set_ylabel(labels[0], fontsize=12)
    axes[0].tick_params(axis='both', labelsize=12)

    axes[0].legend(frameon=False, fontsize=12)

    axes[1].errorbar(centers, np.array(centers)-np.array(centers), color='black', linestyle='--', label='targets')
    axes[1].errorbar(centers, np.array(targets_mean)-np.array(centers), yerr=targets_dev, color='royalblue', marker='o', markersize=4, linestyle='none', label='predictions', alpha=0.8)

    axes[1].set_xlim(*bounds)
    axes[1].set_xlabel(labels[1], fontsize=12)
    axes[1].set_ylabel('Residual', fontsize=12)
    axes[1].set_ylim(*ylim)
    axes[1].tick_params(axis='both', labelsize=12)

In [83]:
BINS = 30

fig, axes = plt.subplots(4,2, gridspec_kw={'height_ratios': [4,1,4,1], 'wspace': 0.1}, figsize=(8,9), layout='constrained')

calibration_curve(axes[:2,0], pred_test[0], testing_datasets[0].y, ['$a_1$', '$\hat{{a}}_1$'], [-1.1e-2,3e-3], BINS, [-1e-3,1e-3])
calibration_curve(axes[:2,1], pred_test[1], testing_datasets[1].y, ['$a_2$', '$\hat{{a}}_2$'], [-6e-4,1.2e-3], BINS, [-3e-4,3e-4])
calibration_curve(axes[2:,0], pred_test[2], testing_datasets[2].y, ['$a_3$', '$\hat{{a}}_3$'], [-3.5e-5,1.2e-6], BINS, [-3e-6,3e-6])
calibration_curve(axes[2:,1], pred_test[3], testing_datasets[3].y, ['$a_4$', '$\hat{{a}}_4$'], [-2e-7,3.6e-6], BINS, [-3e-7,3e-7])

plt.savefig('taylr_calibration.pdf')

In [84]:
# l1 = vector.array({"pt": events_test.kinematics['l1_pt'], "eta": events_test.kinematics['l1_eta'], "phi": events_test.kinematics['l1_phi'], "energy": events_test.kinematics['l1_energy']})
# l2 = vector.array({"pt": events_test.kinematics['l2_pt'], "eta": events_test.kinematics['l2_eta'], "phi": events_test.kinematics['l2_phi'], "energy": events_test.kinematics['l2_energy']})
# l3 = vector.array({"pt": events_test.kinematics['l3_pt'], "eta": events_test.kinematics['l3_eta'], "phi": events_test.kinematics['l3_phi'], "energy": events_test.kinematics['l3_energy']})
# l4 = vector.array({"pt": events_test.kinematics['l4_pt'], "eta": events_test.kinematics['l4_eta'], "phi": events_test.kinematics['l4_phi'], "energy": events_test.kinematics['l4_energy']})
# m4l = (l1 + l2 + l3 + l4).mass
# xobs = m4l
# bins = 41
# bounds = [180,1000]

mtzz = events_test.kinematics['zz_mt']
xobs = mtzz
bins = 31
bounds = [270, 1200]

In [85]:
c6_m = -20
c6_p = +20
c6_vals = [c6_m, c6_p]

c6_mod = c6.Modifier(baseline=msq.Component.SBI, events=events_test, c6_values=[-20,-10,0,10,20])
w_c6, p_c6 = c6_mod.modify(c6_vals)

In [86]:
w_c6_m, p_c6_minus = w_c6[:,0], p_c6[:,0]
w_c6_p, p_c6_plus = w_c6[:,1], p_c6[:,1]

In [87]:
coefficients = np.concatenate([np.ones_like(w_c6_m)[:,np.newaxis], pred_test[0][:,np.newaxis], pred_test[1][:,np.newaxis], pred_test[2][:,np.newaxis], pred_test[3][:,np.newaxis]], axis=1)

reweight_m = np.apply_along_axis(lambda x: np.polyval(x, c6_m), 1, coefficients[:, ::-1])
reweight_p = np.apply_along_axis(lambda x: np.polyval(x, c6_p), 1, coefficients[:, ::-1])

In [89]:
lw = 1.25

h_true_p = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_true_m = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_true_p.fill(xobs, weight=p_c6_plus)
h_true_m.fill(xobs, weight=p_c6_minus)

p_sm = events_test.weights / events_test.weights.sum()
h_sm = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_sm.fill(xobs, weight=p_sm)

p_rwt_plus = p_sm * reweight_p / np.sum(p_sm * reweight_p)
h_reweight_plus = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_reweight_plus.fill(xobs, weight=p_rwt_plus)

p_rwt_minus = p_sm * reweight_m / np.sum(p_sm * reweight_m)
h_reweight_minus = hist.Hist(hist.axis.Regular(bins, *bounds), storage=hist.storage.Weight())
h_reweight_minus.fill(xobs, weight=p_rwt_minus)

fig, (ax1, ax2, ax3) = plt.subplots(3,1,gridspec_kw={'height_ratios': [2, 1, 1]},figsize=(5,6), layout='constrained', sharex=True)
plt.subplots_adjust(wspace=0, hspace=0)

ax1.stairs(h_sm.values(), h_sm.axes[0].edges, color='black', linestyle='--', linewidth=lw, label='$\\mathrm{SM}$')
ax1.stairs(h_reweight_plus.values(), h_reweight_plus.axes[0].edges, color='cornflowerblue',  linestyle='-', linewidth=lw, label=f'$\\mathrm{{SM}} \\rightarrow\\ (C_H = +{c6_p})\\ \\mathrm{{(NSBI)}}$')
ax1.stairs(h_true_p.values(), h_true_p.axes[0].edges, color='blue', linestyle='--', linewidth=lw, label=f'$C_H = +{c6_p}\\ (\\mathrm{{MC}})$')
ax1.stairs(h_reweight_minus.values(), h_reweight_plus.axes[0].edges, color='salmon',  linestyle='-', linewidth=lw, label=f'$\\mathrm{{SM}} \\rightarrow\\ (C_H = {c6_m})\\ \\mathrm{{(NSBI)}}$')
ax1.stairs(h_true_m.values(), h_true_m.axes[0].edges, color='red', linestyle='--', linewidth=lw, label=f'$C_H = {c6_m}\\ (\\mathrm{{MC}})$')

ax1.legend(frameon=False, fontsize=10)

ax1.set_yscale('log')
ax1.set_ylabel('$\\mathrm{Fraction\\ of\\ events}$', loc='top', fontsize=15)
ax1.set_ylim(1e-5, 1.0)
ax1.yaxis.set_ticks([1e-4, 1e-3, 1e-2, 1e-1, 1])

y = h_sm.values()
yerr = np.sqrt(h_sm.variances())
# ax2.fill_between(h_true_p.axes[0].centers, (y-yerr)/y, (y+yerr)/y, linewidth=0, color='black', alpha=0.1, step='mid')
ax2.stairs(h_sm.values()/h_sm.values(), h_sm.axes[0].edges, color='black', linewidth=lw, linestyle='--')

ax2.stairs(h_reweight_plus.values()/h_sm.values(), h_reweight_plus.axes[0].edges, color='cornflowerblue', linestyle='-', linewidth=lw)
ax2.stairs(h_true_p.values()/h_sm.values(), h_true_p.axes[0].edges, color='blue', linestyle='--', linewidth=lw)
ax2.stairs(h_reweight_minus.values()/h_sm.values(), h_reweight_minus.axes[0].edges, color='salmon', linestyle='-', linewidth=lw)
ax2.stairs(h_true_m.values()/h_sm.values(), h_true_m.axes[0].edges, color='red', linestyle='--', linewidth=lw)

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)
ax3.tick_params(labelsize=12)

ax2.set_xlim(180,1000)
ax2.tick_params(top=True, labeltop=False)
ax2.set_xlabel('')
ax2.set_xticklabels([])
ax2.set_ylabel('$p(x | C_H) / p_{ \\mathrm{SM} }(x)$', fontsize=15)
ax2.set_ylim(0.90,1.20)
ax2.yaxis.set_ticks([0.95,1.0,1.15])

ax3.stairs(h_true_p.values()/h_true_p.values(), h_true_p.axes[0].edges, color='lightgrey', linewidth=lw, linestyle='--')
ax3.stairs(h_reweight_plus.values()/h_true_p.values(), h_reweight_plus.axes[0].edges, color='cornflowerblue', linewidth=lw)
ax3.stairs(h_reweight_minus.values()/h_true_m.values(), h_reweight_minus.axes[0].edges, color='salmon', linewidth=lw)

ax3.set_xlabel('$m_{ZZ}\\ \\mathrm{[GeV]}$', loc='right', fontsize=15)
ax3.set_xlim(bounds[0], bounds[1])
ax3.set_xscale('linear')
# ax3.xaxis.set_ticks([180, 250, 500, 750, 1000])

ax3.set_ylabel('$\\mathrm{NSBI} / \\mathrm{MC}$', loc='center', fontsize=15)
ax3.set_ylim(0.96,1.04)
ax3.yaxis.set_ticks([0.98,1.0,1.02])
ax3.tick_params(top=True, labeltop=False)

plt.tight_layout()
plt.subplots_adjust(hspace=0)

plt.savefig('taylr_reweight.pdf', bbox_inches='tight')
plt.close()